# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides step-by-step instructions for loading, exploring, and analyzing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema and is accessible via a public URL.

- **Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed (uncomment if running in Colab or a new environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets and their fields, using each entity's `@id`.

Croissant datasets may contain one or more _record sets_ (tables/files) each with fields corresponding to columns. To get all available record sets and their field IDs:


In [ ]:
# List all record sets (@id and name)
print("Record sets in dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"  - @id: {record_set.id} | name: {record_set.name}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"        ▹ @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame, referencing record set and field `@id`s.

First, we will extract the list of record set `@id`s:

In [ ]:
# Capture all record_set @id strings
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_set_ids)

# Load all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs_id} with shape {df.shape}")

# Show available columns (field @ids) for the first table
if record_set_ids:
    print(f"Columns for {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering, normalization, and grouping—using fields referenced by their `@id`.

Below, select a numeric field and a group field for demonstration. Update the field `@id`s as needed for your EDA.

In [ ]:
# Choose your main record set (table) for analysis
main_record_set_id = record_set_ids[0]  # e.g., 'cr:recordset/0...'
df = dataframes[main_record_set_id]
print(f"Analyzing record set: {main_record_set_id}")

# Identify a numeric field and a grouping field by inspecting available field @ids:
print("Available columns (field @ids):", list(df.columns))

# Example (replace with true field @ids as needed):
# For this demonstration, let's pick likely IDs (you may need to adjust):
# e.g. numeric_field_id = 'cr:field/mw', group_field_id = 'cr:field/accession'

numeric_field_id = df.select_dtypes(include=[float, int]).columns[0] if not df.select_dtypes(include=[float, int]).empty else df.columns[0]
print(f"Chosen numeric field (@id): {numeric_field_id}")

threshold = df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
if threshold is not None:
    print(f"Using threshold (75th percentile): {threshold}")

    # Filter records with the numeric field greater than threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered data shape (> {threshold}): {filtered_df.shape}")

    # Normalize the numeric field (standard score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group_field if present
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype == object:
            group_field_id = col
            print(f"Grouping by: {group_field_id}")
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric fields found to filter and normalize.")

## 5. Visualization
Visualize data distributions and relationships using the DataFrames loaded above.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if threshold is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.2f})')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.legend()
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,4))
        # Show group means for filtered data
        topk = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
        sns.barplot(data=topk, x=group_field_id, y=numeric_field_id)
        plt.title(f"Top 10 {group_field_id}s by mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We successfully used the Croissant schema to load, explore, and visualize the dataset.
- All entities (record sets, fields) were referenced by their `@id` for clarity and reproducibility.
- You can extend this analysis by exploring relationships between more fields, joining record sets, or applying advanced machine learning using the loaded DataFrames.


_This notebook was generated using `mlcroissant` for a FAIR-compliant workflow._